## Aula 4 – Atividade Prática  
### Análise Sintática e Extração de Relações com spaCy

#### Objetivo da atividade

- visualizar análises sintáticas reais produzidas por um parser automático;
- interpretar grafos de dependência sintática;
- identificar sujeito, verbo, objeto e modificadores;
- compreender o papel da análise sintática na extração de relações.

⚠️ **Importante:**  
O objetivo não é treinar um parser, nem focar em programação, mas compreender como a sintaxe aparece na prática em sistemas reais de PLN.


### Instalação do spaCy

In [1]:
# Instalação da biblioteca spaCy
# Necessária apenas em ambientes como Google Colab
!pip -q install spacy

### Download do modelo de português

In [2]:
# Download do modelo de língua portuguesa
# Este modelo inclui:
# - tokenização
# - POS tagging
# - parser de dependências
!python -m spacy download pt_core_news_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 122.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Carregando o pipeline

In [3]:
import spacy
# Carrega o modelo de linguagem para português
nlp = spacy.load("pt_core_news_sm")
print("Pipeline carregado:", nlp.pipe_names)

Pipeline carregado: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


Antes de analisar frases, lembre-se dos conceitos discutidos em aula:

- 🔴 **Núcleo (head):** geralmente o verbo principal da sentença.
- ➡️ **Dependentes:** palavras ligadas ao núcleo.
- 🧩 **Relações sintáticas:** sujeito, objeto, modificador etc.

Nos próximos exemplos, observe:
- qual palavra é o ROOT da frase;
- quem é o sujeito e o objeto;
- como modificadores são conectados.


### Identificando Relações Específicas (nsubj, obj, root)

Termos técnicos:
- ROOT: O núcleo da sentença (geralmente o verbo principal);
- nsubj: Sujeito nominal (quem realiza a ação);
- obj: Objeto direto (quem recebe a ação);
- det: Determinante (artigos);
- case: Um marcador de caso gramatical, geralmente uma preposição ou pós-posição;
- nmod: Um modificador nominal; um substantivo que modifica outro substantivo;
- punct: Sinal de pontuação;
- amod: Um modificador adjetival; um adjetivo que modifica um substantivo;
- advmod: Um modificador adverbial; um advérbio que modifica um verbo, adjetivo ou outro advérbio.

### Exemplo 1: Frase simples

In [4]:
from spacy import displacy

texto = "O aluno leu o livro com atenção."
doc = nlp(texto)

print("Dependências encontradas:\n")
for token in doc:
    print(f"{token.text:<12} --({token.dep_})--> {token.head.text}")

Dependências encontradas:

O            --(det)--> aluno
aluno        --(nsubj)--> leu
leu          --(ROOT)--> leu
o            --(det)--> livro
livro        --(obj)--> leu
com          --(case)--> atenção
atenção      --(nmod)--> livro
.            --(punct)--> leu


### Visualização do grafo

In [5]:
displacy.render(
    doc,
    style="dep",
    jupyter=True,
    options={"distance": 100}
)

### Interpretação do Exemplo 1

Responda:

1. Qual palavra aparece como **ROOT** da sentença?
2. Quem é o **sujeito** do verbo principal?
3. Qual é o **objeto direto**?
4. A expressão *“com atenção”* modifica:
   - o verbo *leu* ou
   - o substantivo *livro*?

Relacione suas respostas com os conceitos de:
- núcleo,
- dependência,
- modificador.


### Exemplo 2 – Ambiguidade Sintática

Agora vamos analisar uma frase estruturalmente ambígua,
discutida em aula teórica.

A frase admite mais de uma interpretação possível.


In [6]:
texto = "O cientista viu a estrela com o telescópio."
doc = nlp(texto)

print("Dependências encontradas:\n")
for token in doc:
    print(f"{token.text:<14} --({token.dep_})--> {token.head.text}")


Dependências encontradas:

O              --(det)--> cientista
cientista      --(nsubj)--> viu
viu            --(ROOT)--> viu
a              --(det)--> estrela
estrela        --(obj)--> viu
com            --(case)--> telescópio
o              --(det)--> telescópio
telescópio     --(nmod)--> estrela
.              --(punct)--> viu


### Visualização da ambiguidade

In [8]:
displacy.render(
    doc,
    style="dep",
    jupyter=True,
    options={"distance": 110}
)

### Discussão – Ambiguidade

1. O sintagma *“com o telescópio”* está ligado a qual palavra?
2. Essa ligação corresponde a qual interpretação da frase?
3. O parser poderia ter escolhido outra estrutura?
4. Como um humano decide qual interpretação é a correta?

➡️ Aqui vemos o parser **tomando uma decisão estrutural**
com base em padrões aprendidos.


### Extração de Relações: Quem fez o quê?

A análise sintática permite extrair informações estruturadas do texto.

Vamos observar:
- quem realizou a ação;
- qual foi a ação;
- quem recebeu a ação.


### Exemplo 3: Evento e participantes

In [9]:
texto = "A empresa adquiriu a startup por 10 milhões."
doc = nlp(texto)

for token in doc:
    print(f"{token.text:<12} --({token.dep_})--> {token.head.text}")

displacy.render(doc, style="dep", jupyter=True, options={"distance": 110})

A            --(det)--> empresa
empresa      --(nsubj)--> adquiriu
adquiriu     --(ROOT)--> adquiriu
a            --(det)--> startup
startup      --(obj)--> adquiriu
por          --(case)--> 10
10           --(obl)--> adquiriu
milhões      --(flat)--> 10
.            --(punct)--> adquiriu


Com base no grafo acima, é possível extrair:

- 🧑‍💼 Quem comprou? → empresa
- 🏢 O que foi comprado? → startup
- 💰 Qual o valor? → 10 milhões

Essas informações podem ser armazenadas em:
- bancos de dados,
- grafos de conhecimento,
- sistemas de consulta estruturada.

É por isso que a análise de dependências é central em tarefas de
**extração de informação**.


### Conexão com o PLN moderno

Modelos como BERT ou GPT não constroem explicitamente
grafos de dependência para funcionar.

No entanto:
- eles aprendem regularidades sintáticas implicitamente;
- mapas de atenção frequentemente refletem dependências sintáticas;
- a sintaxe continua “presente”, embora não explícita.

Estudar análise sintática ajuda a:
- interpretar modelos complexos,
- entender erros,
- extrair conhecimento estruturado.


### Exercício

Analise a frase:

> *A inteligência artificial transforma o mercado de trabalho rapidamente.*

1. Identifique o núcleo da sentença.
2. Identifique sujeito e objeto.
3. O advérbio *“rapidamente”* modifica qual palavra?
4. Como essa informação poderia ser usada em uma aplicação real?

➡️ Utilize o parser para confirmar suas hipóteses.
